In [0]:
# configure paths
catalog = "global_mart_retail_dev"
bronze_table = "superstore_orders"
silver_table = "products"

# table path
bronze_path = f"{catalog}.bronze.{bronze_table}"
silver_path = f"{catalog}.silver.{silver_table}"

print(f"Bronze table path: {bronze_path}")
print(f"Silver table path: {silver_path}")

In [0]:
# import libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
# read bronze data 
bronze_df = spark.read.table(bronze_path)

# creating silver table for products 
df_clean_product = (         
    bronze_df.select(
        trim(col("Product_ID")).alias("product_id"),
        trim(col("Category")).alias("category"),
        trim(col("Sub_Category")).alias("sub_category"),
        # Enhanced product_name cleaning:
        # 1. Replace escaped quotes with single quote
        # 2. Normalize multiple spaces to single space
        # 3. Trim leading/trailing whitespace
        regexp_replace(
            regexp_replace(
                trim(col("Product_Name")),
                '\\"', "'"  # Replace escaped quotes with single quote
            ),
            '\\s+', ' '  # Replace multiple spaces with single space
        ).alias("product_name")
    ).dropDuplicates(["product_id"])                               
)

print(f"total count: {df_clean_product.count()}") # count after dedeuplication


display(df_clean_product.limit(5))



In [0]:
#  Log data quality metrics
print(f"Total products processed: {bronze_df.select('Product_ID').distinct().count()}")
print(f"After cleaning: {df_clean_product.count()}")

In [0]:
# generate hash on cleaned values to track scd type 2

df_product_stage = (
    df_clean_product
    .withColumn("product_hash", sha2(concat_ws("|",col("product_id"),col("category"),col("sub_category"),col("product_name")),256))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to", lit("9999-12-31").cast("timestamp"))
    .withColumn("is_current_flag",lit(True))
    .withColumn("load_timestamp", current_timestamp())
)

display(df_product_stage.limit(5))

In [0]:
# Create the silver products table if it does not exist
spark.sql(
    """
        CREATE TABLE IF NOT EXISTS global_mart_retail_dev.silver.products
        (
            product_id string,
            category string,
            sub_category string,
            product_name string,
            product_hash string not null,
            valid_from timestamp not null,
            valid_to timestamp,
            is_current_flag boolean not null,
            load_timestamp timestamp not null
        )
        USING DELTA
    """
)



# Set up SCD Type 2 merge for product table
product_table = DeltaTable.forName(spark,"global_mart_retail_dev.silver.products")
(
    product_table.alias("target")
    .merge(
        df_product_stage.alias("source"),
        "target.product_id = source.product_id AND target.is_current_flag = true"
    )
    # Expire current records when any attributes have changed
    .whenMatchedUpdate(
        condition="target.product_hash <> source.product_hash",
        set={
            "valid_to": "current_timestamp()",
            "is_current_flag": "false"
        }
    )
    # Insert new records or updated records
    .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "category": "source.category",
            "sub_category": "source.sub_category",
            "product_name": "source.product_name",
            "product_hash": "source.product_hash",
            "valid_from": "source.valid_from",
            "valid_to": "source.valid_to",
            "is_current_flag": "source.is_current_flag",
            "load_timestamp": "source.load_timestamp"
        }
    )
    .execute()
)


In [0]:
%sql
-- sanity check 
select * from global_mart_retail_dev.silver.products

In [0]:
%sql 
select product_id, count(*) from global_mart_retail_dev.silver.products
group by product_id


In [0]:
# # Standardizing column names to be snakescape
# df_clean = df_bronze.select(
#     col("Row_ID").alias("row_id"),
#     col("Order_ID").alias("order_id"),
#     col("Order_Date").alias("order_date_string"), # Keeping original temporarily
#     col("Ship_Date").alias("ship_date_string"),   # Keeping original temporarily
#     col("Ship_Mode").alias("ship_mode"),
#     col("Customer_ID").alias("customer_id"),
#     col("Customer_Name").alias("customer_name"),
#     col("Segment").alias("segment"),
#     col("Country").alias("country"),
#     col("City").alias("city"),
#     col("State").alias("state"),
#     col("Postal_Code").alias("postal_code"),
#     col("Region").alias("region"),
#     col("Product_ID").alias("product_id"),
#     col("Category").alias("category"),
#     col("Sub_Category").alias("sub_category"),
#     col("Product_Name").alias("product_name"),
#     col("Sales").alias("sales"),
#     col("Quantity").alias("quantity"),
#     col("Discount").alias("discount"),
#     col("Profit").alias("profit")
# )

# # Fix Data Types (Date Casting) 
# # Convert date from string like '11/08/2016' to date format.
# df_silver = df_clean.withColumn("order_date", to_date(col("order_date_string"), "M/d/yyyy")) \
#                     .withColumn("ship_date", to_date(col("ship_date_string"), "M/d/yyyy")) \
#                     .drop("order_date_string", "ship_date_string") # Drop the string versions of dates


# display(df_silver.limit(5))